In [ ]:
## Heatmap for minimum hive bee population over gamma_T and epsilon_T
# y-axis: gamma_T in [0.02, 0.09]
# x-axis: epsilon_T in [0.9, 2.1]

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde

# Fixed Parameters
phi_B = 0.018
phi_H = 0.007
epsilon_B = 2000
b = 500
k = 5000
a = 500
tau = 12
gamma_B = 0.003
gamma_H = 0.05
delta_1 = 0.06
delta_2 = 0.06
chi = 600
sigma = 300
theta = 105
kappa = 1/9

# Initial conditions
B0 = 10000
H0 = 20000
f0 = 1000
T0 = 2
V0 = 1

# Simulation setup
STARTTIME = 0
STOPTIME = 365
DT = 0.02
eps = 1e-6
H_crit = 1000
lam = 120


def build_and_run(epsilon_T_val, gamma_T_val):
    # Compute I0 from constant past assumption
    q0 = delta_1 * T0 / (a + B0) + delta_2 * V0 / (a + B0)
    I0 = q0 * tau

    def pos(x):
        return (x + Abs(x)) / 2

    # States
    IDX_B = 0
    IDX_H = 1
    IDX_T = 2
    IDX_V = 3
    IDX_f = 4
    IDX_I = 5

    B = y(IDX_B)
    H = y(IDX_H)
    Tm = y(IDX_T)
    V = y(IDX_V)
    f = y(IDX_f)
    I = y(IDX_I)

    B_tau = y(IDX_B, t - tau)
    T_tau = y(IDX_T, t - tau)
    V_tau = y(IDX_V, t - tau)

    B_pos = pos(B)
    H_pos = pos(H)
    T_pos = pos(Tm)
    V_pos = pos(V)
    f_pos = pos(f)
    I_pos = pos(I)
    B_tau_pos = pos(B_tau)
    T_tau_pos = pos(T_tau)
    V_tau_pos = pos(V_tau)

    S = (f_pos / (b + f_pos)) * (H_pos / (k + H_pos))
    infestation = delta_1 * (B_pos / (a + B_pos)) * T_pos
    infection = delta_2 * (B_pos / (a + B_pos)) * V_pos

    q_now = delta_1 * T_pos / (a + B_pos) + delta_2 * V_pos / (a + B_pos)
    q_tau = delta_1 * T_tau_pos / (a + B_tau_pos) + delta_2 * V_tau_pos / (a + B_tau_pos)
    dI = q_now - q_tau

    exp_B = exp(-gamma_B * tau - I_pos)
    Phi = chi + sigma * cos(2 * pi * (t - theta) / 365)

    dB = epsilon_B * S - infestation - infection - gamma_B * B_pos - kappa * exp_B * B_tau_pos
    dH = kappa * exp_B * B_tau_pos - gamma_H * H_pos
    dT = epsilon_T_val * infestation - gamma_T_val * T_pos
    dV = epsilon_T_val * infection - gamma_T_val * V_pos
    df = Phi * (H_pos / (k + H_pos)) - phi_B * B_pos - phi_H * H_pos

    # Stabilization/projection
    dB += lam * (B_pos - B)
    dH += lam * (H_pos - H)
    dT += lam * (T_pos - Tm)
    dV += lam * (V_pos - V)
    df += lam * (f_pos - f)
    dI += lam * (I_pos - I)

    DDE = jitcdde([dB, dH, dT, dV, df, dI])

    def history(_s):
        return [B0, H0, T0, V0, f0, I0]

    DDE.past_from_function(history)
    DDE.step_on_discontinuities()
    DDE.set_integration_parameters(
        atol=1e-8,
        rtol=1e-6,
        min_step=1e-12,
        max_step=DT,
        first_step=DT,
    )

    t0 = float(DDE.t)
    start = max(STARTTIME + eps, t0)
    times = np.arange(start, STOPTIME + DT / 2, DT)

    minH = H0
    for tt in times:
        state = DDE.integrate(float(tt))
        Hval = float(state[IDX_H])
        if Hval < minH:
            minH = Hval

    return minH


# Sweep ranges
N = 20
epsilon_vals = np.linspace(0.9, 2.1, 20)   
gamma_vals = np.linspace(0.02, 0.09, 20)  

minH_map = np.zeros((len(gamma_vals), len(epsilon_vals)))

for iG, gamma_val in enumerate(gamma_vals):
    for iE, epsilon_val in enumerate(epsilon_vals):
        minH_map[iG, iE] = build_and_run(epsilon_val, gamma_val)

In [ ]:
# Plot
#line_color = (230 / 255, 97 / 255, 0 / 255)

fig, ax = plt.subplots(figsize=(5,4))

base = plt.colormaps["viridis"]
viridis_light = LinearSegmentedColormap.from_list(
    "viridis_light", base(np.linspace(0.22, 1.00, 256))
)

nG, nE = minH_map.shape
E_edges = np.linspace(epsilon_vals[0], epsilon_vals[-1], nE + 1)
G_edges = np.linspace(gamma_vals[0], gamma_vals[-1], nG + 1)
Ec = 0.5 * (E_edges[:-1] + E_edges[1:])
Gc = 0.5 * (G_edges[:-1] + G_edges[1:])
EEc, GGc = np.meshgrid(Ec, Gc)

im = ax.pcolormesh(
    E_edges,
    G_edges,
    minH_map,
    shading="flat",
    cmap=viridis_light,
    edgecolors=(1, 1, 1, 0.7),
    linewidth=0.6,
    antialiased=False,
    zorder=1,
)

cbar = fig.colorbar(im, ax=ax)
#cbar.set_label("Minimum hive bee population", fontsize=15)
cbar.ax.tick_params(labelsize=18, width=1.6, length=6, colors='black')
cbar.set_ticks([20000, 10000, 1000])
cbar.set_ticklabels(["20,000", "10,000", "1,000"])

ax.set_xlim(epsilon_vals[0], epsilon_vals[-1])
ax.set_ylim(gamma_vals[0], gamma_vals[-1])
ax.margins(x=0, y=0)

ax.set_xlabel("$\\epsilon_T$", fontsize=18)
ax.set_ylabel("$\\gamma_T$", fontsize=18)
ax.tick_params(axis='both', labelsize=18, width=1.6, length=6, colors='black')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks([1, 2.1])
ax.set_yticks([0.02, 0.035, 0.09])
ax.set_yticklabels(["0.02", "0.035", "0.09"])

ax.text(-0.13, 1.15, "(a)", transform=ax.transAxes, color='black', fontsize=20, va='top', ha='left')
plt.savefig("heatmap_1a.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
## Heatmap for minimum hive bee population over delta_1 and delta_2
# y-axis: delta_2 in [0.01, 0.1]
# x-axis: delta_1 in [0.01, 0.1]

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde

# Fixed parameters
phi_B = 0.018
phi_H = 0.007
epsilon_B = 2000
b = 500
k = 5000
a = 500
tau = 12
gamma_B = 0.003
gamma_H = 0.05
epsilon_T = 1
gamma_T = 1/24
chi = 600
sigma = 300
theta = 105
kappa = 1/9

# Initial conditions
B0 = 10000
H0 = 20000
f0 = 1000
T0 = 2
V0 = 1

# Simulation setup
STARTTIME = 0
STOPTIME = 365
DT = 0.02
eps = 1e-6
lam = 120


def build_and_run(delta_1_val, delta_2_val):
    q0 = delta_1_val * T0 / (a + B0) + delta_2_val * V0 / (a + B0)
    I0 = q0 * tau

    def pos(x):
        return (x + Abs(x)) / 2

    IDX_B = 0
    IDX_H = 1
    IDX_T = 2
    IDX_V = 3
    IDX_f = 4
    IDX_I = 5

    B = y(IDX_B)
    H = y(IDX_H)
    Tm = y(IDX_T)
    V = y(IDX_V)
    f = y(IDX_f)
    I = y(IDX_I)

    B_tau = y(IDX_B, t - tau)
    T_tau = y(IDX_T, t - tau)
    V_tau = y(IDX_V, t - tau)

    B_pos = pos(B)
    H_pos = pos(H)
    T_pos = pos(Tm)
    V_pos = pos(V)
    f_pos = pos(f)
    I_pos = pos(I)
    B_tau_pos = pos(B_tau)
    T_tau_pos = pos(T_tau)
    V_tau_pos = pos(V_tau)

    S = (f_pos / (b + f_pos)) * (H_pos / (k + H_pos))
    infestation = delta_1_val * (B_pos / (a + B_pos)) * T_pos
    infection = delta_2_val * (B_pos / (a + B_pos)) * V_pos

    q_now = delta_1_val * T_pos / (a + B_pos) + delta_2_val * V_pos / (a + B_pos)
    q_tau = delta_1_val * T_tau_pos / (a + B_tau_pos) + delta_2_val * V_tau_pos / (a + B_tau_pos)
    dI = q_now - q_tau

    exp_B = exp(-gamma_B * tau - I_pos)
    Phi = chi + sigma * cos(2 * pi * (t - theta) / 365)

    dB = epsilon_B * S - infestation - infection - gamma_B * B_pos - kappa * exp_B * B_tau_pos
    dH = kappa * exp_B * B_tau_pos - gamma_H * H_pos
    dT = epsilon_T * infestation - gamma_T * T_pos
    dV = epsilon_T * infection - gamma_T * V_pos
    df = Phi * (H_pos / (k + H_pos)) - phi_B * B_pos - phi_H * H_pos

    dB += lam * (B_pos - B)
    dH += lam * (H_pos - H)
    dT += lam * (T_pos - Tm)
    dV += lam * (V_pos - V)
    df += lam * (f_pos - f)
    dI += lam * (I_pos - I)

    dde = jitcdde([dB, dH, dT, dV, df, dI])

    def history(_s):
        return [B0, H0, T0, V0, f0, I0]

    dde.past_from_function(history)
    dde.step_on_discontinuities()
    dde.set_integration_parameters(
        atol=1e-8,
        rtol=1e-6,
        min_step=1e-12,
        max_step=DT,
        first_step=DT,
    )

    t0 = float(dde.t)
    start = max(STARTTIME + eps, t0)
    times = np.arange(start, STOPTIME + DT / 2, DT)

    minH = H0
    for tt in times:
        state = dde.integrate(float(tt))
        Hval = float(state[IDX_H])
        if Hval < minH:
            minH = Hval

    return minH


# Sweep ranges
N = 20
delta_1_vals = np.linspace(0.01, 0.1, N)
delta_2_vals = np.linspace(0.01, 0.1, N)

minH_map = np.zeros((len(delta_2_vals), len(delta_1_vals)))
for i2, d2 in enumerate(delta_2_vals):
    for i1, d1 in enumerate(delta_1_vals):
        minH_map[i2, i1] = build_and_run(d1, d2)

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(5, 4))

base = plt.colormaps["viridis"]
viridis_light = LinearSegmentedColormap.from_list(
    "viridis_light", base(np.linspace(0.22, 1.00, 256))
)

n2, n1 = minH_map.shape
d1_edges = np.linspace(delta_1_vals[0], delta_1_vals[-1], n1 + 1)
d2_edges = np.linspace(delta_2_vals[0], delta_2_vals[-1], n2 + 1)

im = ax.pcolormesh(
    d1_edges,
    d2_edges,
    minH_map,
    shading="flat",
    cmap=viridis_light,
    edgecolors=(1, 1, 1, 0.7),
    linewidth=0.6,
    antialiased=False,
    zorder=1,
    
)

cbar = fig.colorbar(im, ax=ax)
#cbar.set_label("Minimum hive bee population", fontsize=15)
cbar.ax.tick_params(labelsize=18, width=1.6, length=6, colors='black')
cbar.set_ticks([20000, 10000, 1000])
cbar.set_ticklabels(["20,000", "10,000", "1,000"])

ax.set_xlim(delta_1_vals[0], delta_1_vals[-1])
ax.set_ylim(delta_2_vals[0], delta_2_vals[-1])
ax.margins(x=0, y=0)
ax.set_xlabel("$\\delta_1$", fontsize=18)
ax.set_ylabel("$\\delta_2$", fontsize=18)
ax.tick_params(axis="both", labelsize=12, width=1.2, length=5, colors="black")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", labelsize=18)
ax.set_xticks([0.01, 0.065, 0.1])
ax.set_yticks([0.01, 0.065, 0.1])
ax.set_xticklabels(["0.01", "0.065", "0.1"])
ax.set_yticklabels(["0.01", "0.065", "0.1"])
ax.text(-0.13, 1.15, "(b)", transform=ax.transAxes, color='black', fontsize=20, va='top', ha='left')

plt.savefig("heatmap_1b.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
## Combine Pics
from PIL import Image, ImageOps
import numpy as np

def autocrop(im, pad=6):
    im = im.convert("RGB")
    arr = np.asarray(im).astype(np.int16)
    mask = np.any(arr < 245, axis=2)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return im
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
    x1 = min(im.size[0], x1 + pad); y1 = min(im.size[1], y1 + pad)
    return im.crop((x0, y0, x1, y1))

def pad_to_height(im, target_h):
    w, h = im.size
    if h == target_h:
        return im
    top = (target_h - h)//2
    bottom = target_h - h - top
    return ImageOps.expand(im, border=(0, top, 0, bottom), fill="white")

# Load and crop 
imB = autocrop(Image.open("heatmap_1a.png"), pad=6)
imH = autocrop(Image.open("heatmap_1b.png"), pad=6)

# Match heights
target_h = max(imB.size[1], imH.size[1])
imB = pad_to_height(imB, target_h)
imH = pad_to_height(imH, target_h)

# Pure white space
gap = 40
space = Image.new("RGB", (gap, target_h), (255, 255, 255))  # WHITE

# Stitch
combined = Image.new("RGB", (imB.size[0] + gap + imH.size[0], target_h), (255, 255, 255))
combined.paste(imB, (0, 0))
combined.paste(space, (imB.size[0], 0))
combined.paste(imH, (imB.size[0] + gap, 0))

combined.save("heatmap_1.png", dpi=(600, 600))
print("Saved: heatmap_1.png")
